# Fase 1: las condiciones del mar en cada avistamiento

Para que el modelo aprenda qué prefiere la ballena, cada registro necesita las condiciones del mar en ese lugar y ese mes. También hacen falta **puntos de fondo**: lugares del mar con los que comparar.

El truco contra el sesgo turístico es que los puntos de fondo no se reparten al azar. Siguen la misma concentración que los registros: muchos cerca de Bahía Solano y Nuquí, pocos mar adentro. Así el modelo compara sitios con un esfuerzo de observación parecido y aprende del mar, no de dónde salen las lanchas (Phillips et al., 2009).

| Variable | Fuente en Earth Engine | Resolución |
|---|---|---|
| Temperatura del mar del mes | `NOAA/CDR/OISST/V2_1` | 0,25° (~28 km) |
| Clorofila (climatología del mes, 2003-2024) | `NASA/OCEANDATA/MODIS-Aqua/L3SMI` | ~4,6 km |
| Profundidad y pendiente del fondo | `NOAA/NGDC/ETOPO1` | ~1,8 km |
| Distancia a la costa | Natural Earth 10 m | vectorial |

**Antes de empezar:** sube `avistamientos_pacifico_co.csv` (sale de la fase 0) y escribe tu proyecto de Google Cloud con Earth Engine en `PROYECTO`.

In [ ]:
!pip -q install earthengine-api pyproj
import ee, json, os, time, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt, shapely
from shapely.geometry import shape
from shapely.ops import unary_union, transform
from scipy.stats import gaussian_kde
from pyproj import Transformer

PROYECTO = "tu-proyecto-de-earth-engine"   # <- cambia esto
ee.Authenticate()
ee.Initialize(project=PROYECTO)

OESTE, ESTE, SUR, NORTE = -82.5, -77.0, 1.4, 7.4
N_FONDO = 10000
rng = np.random.default_rng(42)
ACENTO, NARANJA, TINTA, GRIS = "#0F6E7A", "#E07A2E", "#1C211D", "#9AA19B"
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": GRIS, "xtick.color": TINTA, "ytick.color": TINTA})

## 1. Avistamientos y adelgazamiento
Varios registros del mismo sitio y el mismo mes repiten la misma información, sobre todo la telemetría (una ballena registrada muchas veces). Se deja **uno por celda de ~2 km y por mes**.

In [ ]:
RUTA = "avistamientos_pacifico_co.csv"
if not os.path.exists(RUTA):
    from google.colab import files
    files.upload()
df = pd.read_csv(RUTA, parse_dates=["fecha"])
if "tipo" not in df:
    df["tipo"] = np.where(df.base.astype(str).str.upper().str.replace("_", "") == "MACHINEOBSERVATION",
                          "telemetría", "avistamiento")
df["celda"] = (df.lat / 0.02).round().astype(int).astype(str) + "_" + (df.lon / 0.02).round().astype(int).astype(str)
df["ym"] = df.fecha.dt.to_period("M").astype(str)
pres = df.drop_duplicates(["celda", "ym"]).copy()
pres["presencia"] = 1
print(f"{len(df)} registros -> {len(pres)} presencias tras el adelgazamiento")
print(pres.tipo.value_counts().to_dict())

## 2. Tierra y distancia a la costa

In [ ]:
NE = "ne_10m_land.geojson"
if not os.path.exists(NE):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_land.geojson", NE)
caja = shapely.box(OESTE - 1, SUR - 1, ESTE + 1, NORTE + 1)
tierra = unary_union([shape(f["geometry"]).intersection(caja) for f in json.load(open(NE))["features"]])
a_utm = Transformer.from_crs(4326, 32618, always_xy=True)       # UTM 18N, en metros
tierra_m = transform(a_utm.transform, tierra)
shapely.prepare(tierra_m)

def dist_costa_km(lon, lat):
    x, y = a_utm.transform(np.asarray(lon), np.asarray(lat))
    return shapely.distance(tierra_m, shapely.points(x, y)) / 1000

## 3. Puntos de fondo con corrección del sesgo
Una superficie de esfuerzo (densidad de todos los registros) decide dónde caen los puntos de fondo: 80 % según esa superficie y 20 % repartidos por todo el mar, para que el modelo también vea aguas abiertas. Cada punto de fondo recibe una fecha tomada de las presencias, así comparte su misma distribución de meses y años.

In [ ]:
lon_g, lat_g = np.meshgrid(np.arange(OESTE, ESTE, 0.02), np.arange(SUR, NORTE, 0.02))
lon_g, lat_g = lon_g.ravel(), lat_g.ravel()
mar = ~shapely.contains_xy(tierra, lon_g, lat_g)
lon_g, lat_g = lon_g[mar], lat_g[mar]

kde = gaussian_kde(np.vstack([df.lon, df.lat]), bw_method=0.15)
esfuerzo = kde(np.vstack([lon_g, lat_g]))
p = 0.8 * esfuerzo / esfuerzo.sum() + 0.2 / len(esfuerzo)
idx = rng.choice(len(p), size=N_FONDO, replace=False, p=p / p.sum())
fondo = pd.DataFrame({
    "lon": lon_g[idx] + rng.uniform(-0.01, 0.01, N_FONDO),
    "lat": lat_g[idx] + rng.uniform(-0.01, 0.01, N_FONDO),
    "fecha": rng.choice(pres.fecha.values, N_FONDO),
    "tipo": "fondo", "presencia": 0})

fig, ax = plt.subplots(figsize=(8, 8.5), dpi=150)
ax.tripcolor(lon_g, lat_g, esfuerzo, cmap="Blues", shading="gouraud")
for g in getattr(tierra, "geoms", [tierra]):
    ax.fill(*g.exterior.xy, color="#EFEDE6", ec=GRIS, lw=0.5)
ax.scatter(fondo.lon, fondo.lat, s=1.5, c=TINTA, alpha=0.35, lw=0, label="Fondo")
ax.scatter(pres.lon, pres.lat, s=6, c=NARANJA, lw=0, label="Presencias")
ax.set_xlim(OESTE, ESTE); ax.set_ylim(SUR, NORTE); ax.set_aspect("equal")
ax.legend(loc="lower left", frameon=False)
ax.set_title("Superficie de esfuerzo y puntos de fondo", loc="left", color=TINTA)
plt.savefig("fase1_fondo.png", bbox_inches="tight"); plt.show()

## 4. Variables del mar en Earth Engine
Se agrupan los puntos por año y mes; para cada grupo se arma una imagen con la temperatura de ese mes, la clorofila típica de ese mes, la profundidad y la pendiente, y se lee el valor en cada punto.

In [ ]:
pts = pd.concat([pres[["lon", "lat", "fecha", "tipo", "presencia"]], fondo], ignore_index=True)
pts["fecha"] = pd.to_datetime(pts.fecha)
pts["uid"] = np.arange(len(pts))
pts["año"], pts["mes"] = pts.fecha.dt.year, pts.fecha.dt.month

sst_col = ee.ImageCollection("NOAA/CDR/OISST/V2_1").select("sst")
chl_col = (ee.ImageCollection("NASA/OCEANDATA/MODIS-Aqua/L3SMI").select("chlor_a")
           .filterDate("2003-01-01", "2025-01-01"))
prof = ee.Image("NOAA/NGDC/ETOPO1").select("bedrock").rename("prof")
fijas = prof.addBands(ee.Terrain.slope(prof).rename("pendiente"))
CHL = {m: chl_col.filter(ee.Filter.calendarRange(m, m, "month")).median().rename("chl") for m in range(1, 13)}

def extraer(grupo, año, mes):
    ini = ee.Date.fromYMD(int(año), int(mes), 1)
    sst = sst_col.filterDate(ini, ini.advance(1, "month")).mean().multiply(0.01).rename("sst")
    img = sst.addBands(CHL[int(mes)]).addBands(fijas)
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([r.lon, r.lat]), {"uid": int(r.uid)})
                               for r in grupo.itertuples()])
    out = img.reduceRegions(collection=fc, reducer=ee.Reducer.first(), scale=5000).getInfo()
    return pd.DataFrame([f["properties"] for f in out["features"]])

grupos = list(pts.groupby(["año", "mes"]))
filas = []
for i, ((año, mes), g) in enumerate(grupos, 1):
    for intento in range(3):
        try:
            filas.append(extraer(g, año, mes)); break
        except Exception as e:
            print(f"{año}-{mes:02d}: reintento ({e})"); time.sleep(5)
    if i % 20 == 0 or i == len(grupos):
        print(f"{i}/{len(grupos)} meses procesados")
vars_mar = pd.concat(filas, ignore_index=True)

## 5. Unir y guardar

In [ ]:
tabla = pts.merge(vars_mar, on="uid", how="left")
tabla["dist_costa_km"] = dist_costa_km(tabla.lon, tabla.lat)
tabla["log_chl"] = np.log10(tabla.chl)
tabla["prof"] = -tabla.prof            # metros bajo el nivel del mar, positivo
tabla.loc[tabla.prof < 0, "prof"] = np.nan  # celdas que el raster ve como tierra
cols = ["uid", "presencia", "tipo", "lon", "lat", "fecha", "año", "mes",
        "sst", "chl", "log_chl", "prof", "pendiente", "dist_costa_km"]
tabla[cols].to_csv("variables_fase1.csv", index=False)

print(tabla.groupby("presencia")[["sst", "log_chl", "prof", "pendiente", "dist_costa_km"]].median().round(2))
print()
print("Datos faltantes por variable:")
print(tabla[["sst", "chl", "prof", "pendiente"]].isna().mean().round(3))

## 6. ¿En qué se diferencian las presencias del fondo?
Si las curvas se separan, esa variable ayuda a explicar dónde están las ballenas.

In [ ]:
var = [("sst", "Temperatura del mar (°C)"), ("log_chl", "Clorofila (log10 mg/m³)"),
       ("prof", "Profundidad (m)"), ("pendiente", "Pendiente del fondo (°)"),
       ("dist_costa_km", "Distancia a la costa (km)")]
fig, axs = plt.subplots(1, 5, figsize=(20, 3.8), dpi=150)
for ax, (v, titulo) in zip(axs, var):
    datos = tabla[v].dropna()
    bins = np.linspace(datos.quantile(0.01), datos.quantile(0.99), 40)
    for p, c, n in [(0, GRIS, "Fondo"), (1, ACENTO, "Presencias")]:
        ax.hist(tabla.loc[tabla.presencia == p, v].dropna(), bins=bins, density=True,
                histtype="stepfilled" if p == 0 else "step", color=c, alpha=0.5 if p == 0 else 1, lw=2, label=n)
    ax.set_title(titulo, loc="left", fontsize=11, color=TINTA); ax.set_yticks([])
axs[0].legend(frameon=False)
plt.tight_layout(); plt.savefig("fase1_variables.png", bbox_inches="tight"); plt.show()

## 7. Descargar

In [ ]:
from google.colab import files
for f in ["variables_fase1.csv", "fase1_fondo.png", "fase1_variables.png"]:
    files.download(f)